# Credit Card Customer Journey — Exploratory Data Analysis
**Analyst:** Sagar Kandelkar | **Date:** September 2026
**Data:** Synthetic credit card dataset for portfolio case study

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
applications = pd.read_csv('../data/applications.csv')
customers = pd.read_csv('../data/customers.csv')
transactions = pd.read_csv('../data/transactions.csv')
rewards = pd.read_csv('../data/rewards.csv')

print('Applications:', applications.shape)
print('Customers:', customers.shape)
print('Transactions:', transactions.shape)
print('Rewards:', rewards.shape)

## 2. Application Funnel Analysis

In [ ]:
total_apps = len(applications)
approved = applications[applications['application_status'] == 'approved']
rejected = applications[applications['application_status'] == 'rejected']
activated = approved[approved['activation_date'].notna()]

funnel_data = [total_apps, len(approved), len(activated)]
funnel_labels = ['Applied', 'Approved', 'Activated']

fig, ax = plt.subplots()
bars = ax.barh(funnel_labels, funnel_data, color=['#94a3b8', '#2563eb', '#0f172a'])
for bar, val in zip(bars, funnel_data):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold')
ax.set_title('Application Funnel')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

print(f'Approval Rate: {len(approved)/total_apps*100:.1f}%')
print(f'Activation Rate: {len(activated)/len(approved)*100:.1f}%')

## 3. Customer Segment Distribution

In [ ]:
segment_counts = customers['customer_segment'].value_counts()
colors = ['#0f172a', '#2563eb', '#94a3b8', '#64748b']
segment_counts.plot(kind='pie', autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Customer Segment Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 4. Credit Score vs Approved Limit

In [ ]:
approved_apps = applications[applications['application_status'] == 'approved'].copy()
sns.scatterplot(x='credit_score', y='approved_limit', hue='product_type', data=approved_apps, s=100)
plt.title('Credit Score vs Approved Limit')
plt.xlabel('Credit Score')
plt.ylabel('Approved Limit (INR)')
plt.tight_layout()
plt.show()

## 5. Spend by Merchant Category

In [ ]:
cat_spend = transactions.groupby('merchant_category')['amount'].sum().sort_values(ascending=False)
cat_spend.plot(kind='bar', color='#2563eb')
plt.title('Spend by Merchant Category')
plt.xlabel('Category')
plt.ylabel('Total INR')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Credit Utilization by Customer

In [ ]:
active_customers = customers[customers['active_cards_count'] > 0].copy()
active_customers['utilization_pct'] = (active_customers['total_utilized'] / active_customers['total_credit_limit']) * 100

sns.histplot(active_customers['utilization_pct'], bins=10, kde=True, color='#2563eb')
plt.title('Credit Utilization Distribution')
plt.xlabel('Utilization %')
plt.ylabel('Count')
plt.axvline(active_customers['utilization_pct'].mean(), color='#dc2626', linestyle='--', label=f'Mean: {active_customers["utilization_pct"].mean():.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Average utilization: {active_customers["utilization_pct"].mean():.1f}%')

## 7. Rewards Points Analysis

In [ ]:
reward_summary = rewards.groupby('customer_id').agg(
    total_earned=('points_earned', 'sum'),
    total_redeemed=('redemption_value_inr', 'sum'),
    max_balance=('points_balance', 'max')
).reset_index().merge(customers[['customer_id', 'customer_name', 'customer_segment']], on='customer_id')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='customer_segment', y='total_earned', data=reward_summary, ax=axes[0], palette='Blues')
axes[0].set_title('Points Earned by Segment')
axes[0].set_ylabel('Points Earned')

sns.boxplot(x='customer_segment', y='total_redeemed', data=reward_summary, ax=axes[1], palette='Greens')
axes[1].set_title('Redemption Value by Segment')
axes[1].set_ylabel('INR Redeemed')

plt.tight_layout()
plt.show()

## 8. Transaction Type Split (POS vs Online)

In [ ]:
type_split = transactions.groupby('transaction_type').agg({'amount': ['count', 'sum']})
type_split.columns = ['Count', 'Volume']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
type_split['Count'].plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=['#2563eb', '#dc2626'])
axes[0].set_title('Transaction Count Split')
axes[0].set_ylabel('')

type_split['Volume'].plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2563eb', '#dc2626'])
axes[1].set_title('Transaction Volume Split')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(type_split)

## 9. Key Insights & Recommendations

1. **Application Funnel:** 85% approval rate with 100% activation among approved — strong bottom funnel. Rejection reasons focus on low credit score and income.
2. **Customer Segments:** Platinum and Gold segments dominate — focus retention here. Prospective customers need nurturing.
3. **Credit Utilization:** Average utilization is ~45% — healthy but room for limit enhancement campaigns.
4. **Spend Categories:** Travel and E-commerce drive highest volumes — tailor rewards and offers.
5. **Rewards:** Platinum customers earn and redeem significantly more — ensure premium redemption catalog.
6. **Channel Split:** Online transactions slightly outnumber POS — ensure seamless digital experience.

### Recommendations
- Implement instant approval for customers with credit score >750 and existing bank relationship
- Launch 'spend & earn' campaigns for Travel and E-commerce categories
- Trigger limit enhancement offers for customers with <40% utilization and 6+ months history
- Send personalized redemption nudges to customers with >500 unused points
- Create a win-back journey for prospective customers with rejected applications after 90 days